In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers torchvision torchaudio --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 592.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/

In [5]:
if not os.path.exists('/content/epic-kitchens-55-annotations'):
    !git clone https://github.com/epic-kitchens/epic-kitchens-55-annotations.git /content/epic-kitchens-55-annotations


!dir /content/drive/mydrive/tim_files/

Cloning into '/content/epic-kitchens-55-annotations'...
remote: Enumerating objects: 675, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 675 (delta 93), reused 62 (delta 43), pack-reused 551 (from 1)
Receiving objects: 100% (675/675), 22.53 MiB | 13.52 MiB/s, done.
Resolving deltas: 100% (449/449), done.
dir: cannot access '/content/drive/mydrive/tim_files/': No such file or directory


In [3]:
import os, json, tarfile, glob
from tqdm import tqdm
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torchvision.transforms as T
from transformers import CLIPProcessor, CLIPModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [22]:


BASE_DIR = "/content"


ANNOT_ROOT   = os.path.join(BASE_DIR, "epic-kitchens-55-annotations")
DATA_ROOT    = os.path.join(BASE_DIR, "EPIC-KITCHENS55")
FRAMES_ROOT  = os.path.join(BASE_DIR, "frames")
os.makedirs(FRAMES_ROOT, exist_ok=True)


VIDEO_IDS = [
    "P01_16", "P01_18", "P02_11", "P03_04",
    "P14_01", "P14_02", "P14_03", "P14_04",
    "P14_05", "P14_07", "P14_09"
]

def video_frames_path(video_id: str) -> str:
    """
    Folder that contains the extracted frames for a video.
    Updated to handle the PXX/PXX_XX subfolder structure.
    """
    # Extracts 'P01' from 'P01_16'
    participant_id = video_id.split('_')[0]
    return os.path.join(FRAMES_ROOT, participant_id, video_id)

In [18]:
verb_df

,verb_id,class_key,verbs
0,0,take,"['take', 'grab', 'pick', 'draw', 'get', 'grab-..."
1,1,put,"['put', 'pose', 'put-away', 'place-that', 'pla..."
2,2,open,"['open', 'unzip', 'open-up']"
3,3,close,"['close', 'close-off', 'shut']"
4,4,wash,"['wash', 'sponge', 'lather', 'wash-with', 'rin..."
...,...,...,...
120,120,unfreeze,['unfreeze']
121,121,decide-if,['decide-if']
122,122,let-out,['let-out']
123,123,save,['save']


In [ ]:

VERB_CSV   = os.path.join(ANNOT_ROOT, "EPIC_verb_classes.csv")
NOUN_CSV   = os.path.join(ANNOT_ROOT, "EPIC_noun_classes.csv")
LABEL_CSV  = os.path.join(ANNOT_ROOT, "EPIC_train_action_labels.csv")  

verb_df = pd.read_csv(VERB_CSV)
noun_df = pd.read_csv(NOUN_CSV)

verb_id_to_name = dict(zip(verb_df["verb_id"], verb_df["class_key"]))
noun_id_to_name = dict(zip(noun_df["noun_id"], noun_df["class_key"]))

df_labels = pd.read_csv(LABEL_CSV)

df_labels = df_labels[df_labels["video_id"].isin(VIDEO_IDS)].reset_index(drop=True)

In [ ]:
verb_counts = df_labels["verb_class"].value_counts()
noun_counts = df_labels["noun_class"].value_counts()

TOP_VERBS = verb_counts.head(50).index.tolist()
TOP_NOUNS = noun_counts.head(150).index.tolist()

candidate_actions = []
for v_id in TOP_VERBS:
    v_name = verb_id_to_name.get(v_id, str(v_id))
    for n_id in TOP_NOUNS:
        n_name = noun_id_to_name.get(n_id, str(n_id))
        candidate_actions.append({
            "verb_id": int(v_id),
            "noun_id": int(n_id),
            "text": f"{v_name} {n_name}"
        })

In [ ]:

# CLIP model  we will average frame embeddings.
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

# CLIP expects ImageNet‑normalized 224×224 RGB images
clip_transform = T.Compose([
    T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225])
])

In [ ]:
def get_frame_ids(start_frame: int, stop_frame: int, n_frames: int = 16):
    """Uniformly sample `n_frames` indices between start and stop (inclusive)."""
    if stop_frame <= start_frame:
        stop_frame = start_frame + n_frames
    seg = (stop_frame - start_frame - 1) / n_frames
    ids = []
    for i in range(n_frames):
        s = int(round(seg * i) + start_frame)
        e = int(round(seg * (i + 1)) + start_frame)
        fid = (s + e) // 2
        ids.append(min(fid, stop_frame - 1))
    return ids


def ensure_frames_extracted(video_id: str) -> str:
    """
    Extract the .tar archive for `video_id` if the frames folder does not exist yet.
    Returns the absolute path to the folder that now contains the JPEG frames.
    """
    out_dir = video_frames_path(video_id)
    if os.path.isdir(out_dir) and len(os.listdir(out_dir)) > 0:
        return out_dir 

    tar_path = None
    for root, _, files in os.walk(DATA_ROOT):
        for f in files:
            if f.startswith(video_id) and f.lower().endswith(".tar"):
                tar_path = os.path.join(root, f)
                break
        if tar_path:
            break


    os.makedirs(out_dir, exist_ok=True)
    with tarfile.open(tar_path, "r") as tar:
        tar.extractall(path=out_dir)
    return out_dir


def load_frames(frames_dir: str, start_frame: int, stop_frame: int, n_frames: int = 16):
    """
    Load `n_frames` RGB frames from `frames_dir`, apply CLIP transform,
    and return a tensor of shape [T, 3, 224, 224].
    """
    ids = get_frame_ids(start_frame, stop_frame, n_frames)
    imgs = []
    for fid in ids:
        # Try two common naming conventions
        candidates = [
            os.path.join(frames_dir, f"frame_{fid:010d}.jpg"),
            os.path.join(frames_dir, f"{fid}.jpg")
        ]
        img_path = None
        for p in candidates:
            if os.path.exists(p):
                img_path = p
                break

        if img_path is None:
            # fallback: duplicate last good frame or black image
            if imgs:
                imgs.append(imgs[-1])
            else:
                imgs.append(torch.zeros(3, 224, 224))
            continue

        img = Image.open(img_path).convert("RGB")
        img = clip_transform(img)          # [3, 224, 224]
        imgs.append(img)

    return torch.stack(imgs, dim=0)        # [T, 3, 224, 224]


@torch.no_grad()
def video_embedding(frames_tensor: torch.Tensor):
    """
    Compute a CLIP video embedding by averaging frame embeddings.
    Input: [T, 3, 224, 224] (CPU tensor)
    Output: [1, D] (CPU tensor)
    """
    frames_tensor = frames_tensor.to(device)
    img_emb = clip_model.get_image_features(frames_tensor)   # [T, D]
    img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)   # L2‑norm
    vid_emb = img_emb.mean(dim=0, keepdim=True)              # [1, D]
    return vid_emb.cpu()


@torch.no_grad()
def encode_candidate_texts(candidate_texts):
    """
    Encode all candidate sentences once.
    Returns a tensor [N, D] on CPU.
    """
    batch_size = 256
    all_emb = []
    for i in range(0, len(candidate_texts), batch_size):
        batch = candidate_texts[i:i+batch_size]
        inputs = clip_processor(text=batch, return_tensors="pt", padding=True).to(device)
        txt_emb = clip_model.get_text_features(**inputs)          # [B, D]
        txt_emb = txt_emb / txt_emb.norm(dim=-1, keepdim=True)    # L2‑norm
        all_emb.append(txt_emb.cpu())
    return torch.cat(all_emb, dim=0)   # [N, D]


def select_distractors(video_emb, cand_emb, cand_actions, gt_verb_id, gt_noun_id, top_k=19):
    """
    Return the top‑k distractors (answer + confidence) that are NOT the ground‑truth pair.
    """
    sims = (video_emb @ cand_emb.T).squeeze(0)   # [N]
    sorted_idx = torch.argsort(sims, descending=True)

    distractors = []
    for idx in sorted_idx.tolist():
        act = cand_actions[idx]
        if act["verb_id"] == gt_verb_id and act["noun_id"] == gt_noun_id:
            continue
        distractors.append({
            "answer": act["text"],
            "confidence": f"{sims[idx].item():.4f}"
        })
        if len(distractors) >= top_k:
            break
    return distractors

In [ ]:
candidate_texts = [a["text"] for a in candidate_actions]
cand_emb = encode_candidate_texts(candidate_texts)   # [N, D]


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import tarfile
import glob
import json
from PIL import Image

# Find all participant folders (P01, P02, etc.) under the EPIC folder
participant_folders = sorted(glob.glob(os.path.join(DRIVE_ROOT, "P*")))

for p_path in participant_folders:
    p_id = os.path.basename(p_path)
    tar_files = sorted(glob.glob(os.path.join(p_path, "*.tar")))

    for tar_file in tar_files:
        video_id = os.path.basename(tar_file).replace(".tar", "")

        # Only extract if it's in our target list
        if video_id in TARGET_VIDEOS:
            target_dir = os.path.join(LOCAL_EXTRACT_PATH, p_id, video_id)

            # Skip if already extracted
            if os.path.exists(target_dir) and len(os.listdir(target_dir)) > 0:
                continue

            os.makedirs(target_dir, exist_ok=True)
            with tarfile.open(tar_file, 'r') as tar:
                tar.extractall(path=target_dir)

In [ ]:
results = []

for video_id in tqdm(VIDEO_IDS, desc="Videos"):

    frames_dir = ensure_frames_extracted(video_id)

    df_vid = df_labels[df_labels["video_id"] == video_id]

    for _, row in df_vid.iterrows():
        # ---- basic metadata -------------------------------------------------
        uid            = int(row["uid"])
        participant_id = row["participant_id"]
        start_ts       = row["start_timestamp"]
        stop_ts        = row["stop_timestamp"]
        start_frame    = int(row["start_frame"])
        stop_frame     = int(row["stop_frame"])
        n_frames       = 16
        frame_indices  = get_frame_ids(start_frame, stop_frame, n_frames)

        # ---- load frames & compute video embedding ----------------------------
        frames_tensor = load_frames(frames_dir, start_frame, stop_frame, n_frames)  # [T,3,224,224]
        vid_emb = video_embedding(frames_tensor)                                   # [1,D]

        # ---- ground‑truth verb / noun ----------------------------------------
        gt_verb_id   = int(row["verb_class"])
        gt_noun_id   = int(row["noun_class"])
        gt_verb_name = verb_id_to_name.get(gt_verb_id, str(gt_verb_id))
        gt_noun_name = noun_id_to_name.get(gt_noun_id, str(gt_noun_id))

        # ---- select distractors ------------------------------------------------
        distractors = select_distractors(
            video_emb=vid_emb,
            cand_emb=cand_emb,
            cand_actions=candidate_actions,
            gt_verb_id=gt_verb_id,
            gt_noun_id=gt_noun_id,
            top_k=19
        )

        # ---- build the result entry -----------------------------------------
        entry = {
            "uid": uid,
            "participant_id": participant_id,
            "video_id": video_id,
            "start_timestamp": start_ts,
            "stop_timestamp": stop_ts,
            "start_frame": start_frame,
            "stop_frame": stop_frame,
            "n_frames": n_frames,
            "frame_indices": frame_indices,
            "ground_truth": {
                "verb": gt_verb_id,
                "verb_class": gt_verb_name,
                "noun": gt_noun_id,
                "noun_class": gt_noun_name,
                "narration": row["narration"]
            },
            "distractors_with_confidence": distractors,
            "num_options": len(distractors) + 1,   # 19 + GT
            "model": "CLIP_ViT-B/32"
        }
        results.append(entry)


In [ ]:
output_path = os.path.join(BASE_DIR, "epic_distractors_CLIP_multi2.json")
with open(output_path, "w") as f:
    json.dump(results, f, indent=2)

In [40]:
import random
for i in random.sample(range(len(results)), 3):
    print("\n--- Sample", i, "---")
    print(json.dumps(results[i], indent=2))

# How many unique distractor verbs / nouns did we generate overall?
verb_set = set()
noun_set = set()
for r in results:
    for d in r["distractors_with_confidence"]:
        v, n = d["answer"].split(maxsplit=1)
        verb_set.add(v)
        noun_set.add(n)

print("\nUnique distractor verbs :", len(verb_set))
print("Unique distractor nouns :", len(noun_set))


--- Sample 759 ---
{
  "uid": 5769,
  "participant_id": "P02",
  "video_id": "P02_11",
  "start_timestamp": "00:00:29.28",
  "stop_timestamp": "00:00:30.38",
  "start_frame": 1756,
  "stop_frame": 1822,
  "n_frames": 16,
  "frame_indices": [
    1758,
    1762,
    1766,
    1770,
    1774,
    1778,
    1782,
    1786,
    1790,
    1795,
    1799,
    1803,
    1807,
    1811,
    1815,
    1819
  ],
  "ground_truth": {
    "verb": 0,
    "verb_class": "take",
    "noun": 18,
    "noun_class": "fork",
    "narration": "pick up fork"
  },
  "distractors_with_confidence": [
    {
      "answer": "sharpen risotto",
      "confidence": "0.2700"
    },
    {
      "answer": "shake paella",
      "confidence": "0.2687"
    },
    {
      "answer": "fill spatula",
      "confidence": "0.2675"
    },
    {
      "answer": "pull paella",
      "confidence": "0.2673"
    },
    {
      "answer": "pull risotto",
      "confidence": "0.2670"
    },
    {
      "answer": "sharpen paella",
      

In [ ]:

import json
from tqdm import tqdm


RESULTS_PATH = os.path.join(BASE_DIR, "epic_distractors_CLIP_multi2.json")
with open(RESULTS_PATH, "r") as f:
    results = json.load(f)

print(f"🔎 Loaded {len(results)} action entries for evaluation")

def gt_rank(video_emb, cand_emb, cand_actions, gt_verb_id, gt_noun_id):
    """
    Returns the 1‑based rank of the ground‑truth (verb_id, noun_id) among
    *all* candidate actions sorted by cosine similarity.
    """
    # cosine similarity of the video embedding with every candidate
    sims = (video_emb @ cand_emb.T).squeeze(0)          # [N]

    # sort descending → highest similarity first
    sorted_idx = torch.argsort(sims, descending=True)

    # walk through the sorted list until we hit the GT pair
    for rank, idx in enumerate(sorted_idx.tolist(), start=1):
        act = cand_actions[idx]
        if act["verb_id"] == gt_verb_id and act["noun_id"] == gt_noun_id:
            return rank

    return None


ranks = []                     # list of integer ranks (1‑based)
missing = 0

for entry in tqdm(results, desc="Evaluating"):

    video_id       = entry["video_id"]
    start_frame    = entry["start_frame"]
    stop_frame     = entry["stop_frame"]
    gt_verb_id     = entry["ground_truth"]["verb"]
    gt_noun_id     = entry["ground_truth"]["noun"]

    frames_dir = video_frames_path(video_id)
    frames_tensor = load_frames(frames_dir, start_frame, stop_frame, n_frames=16)

    vid_emb = video_embedding(frames_tensor)   # [1, D] on CPU

    rank = gt_rank(vid_emb, cand_emb, candidate_actions, gt_verb_id, gt_noun_id)
    if rank is None:
        missing += 1
    else:
        ranks.append(rank)


ranks_arr = np.array(ranks)

top1  = np.mean(ranks_arr <= 1) * 100
top5  = np.mean(ranks_arr <= 5) * 100
top10 = np.mean(ranks_arr <= 10) * 100
top20 = np.mean(ranks_arr <= 20) * 100

